# Zomato Discount Cohort Analysis
## Notebook 1 — Data Generation & Cleaning
**Project by: Gunika Sharma**

### Problem Statement
Zomato runs 40–60% discount campaigns to acquire customers.
**Central question:** Are discount-acquired customers actually profitable at 90 days?
Which restaurant categories are subsidising acquisition? At what discount threshold does a customer become permanently coupon-dependent?

---

## Step 1 — Import Libraries

In [16]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully')
print(f'Pandas version : {pd.__version__}')
print(f'NumPy version  : {np.__version__}')

Libraries loaded successfully
Pandas version : 3.0.2
NumPy version  : 2.4.4


## Step 2 — Synthesise Realistic Order Data

No public order-level coupon data exists for Zomato. We generate 10,000 synthetic customer orders that mirror real-world patterns.

**Why synthetic data is valid:** The distribution parameters (discount rates, retention curves, order values) are based on published Zomato/Swiggy investor reports and industry benchmarks. The insight logic is sound regardless of source.

In [17]:
np.random.seed(42)   # for reproducibility — same results every run

N_CUSTOMERS = 3000
N_ORDERS    = 10000

# ── Customer master ──────────────────────────────────────────────────────────
customer_ids = [f'CUST_{str(i).zfill(5)}' for i in range(1, N_CUSTOMERS + 1)]

# Acquisition channel: 60% via discount, 40% organic
acquisition_type = np.random.choice(
    ['discount_acquired', 'organic_acquired'],
    size=N_CUSTOMERS,
    p=[0.60, 0.40]
)

# Acquisition date spread across 12 months
acquisition_dates = pd.to_datetime('2023-01-01') + pd.to_timedelta(
    np.random.randint(0, 365, N_CUSTOMERS), unit='D'
)

customers = pd.DataFrame({
    'customer_id'       : customer_ids,
    'acquisition_type'  : acquisition_type,
    'acquisition_date'  : acquisition_dates,
    'city'              : np.random.choice(
        ['Delhi', 'Mumbai', 'Bengaluru', 'Hyderabad', 'Pune', 'Chennai'],
        size=N_CUSTOMERS,
        p=[0.25, 0.25, 0.20, 0.12, 0.10, 0.08]
    )
})

print(f'Customers created   : {len(customers)}')
print(customers['acquisition_type'].value_counts())

Customers created   : 3000
acquisition_type
discount_acquired    1777
organic_acquired     1223
Name: count, dtype: int64


In [18]:
# ── Order-level data ─────────────────────────────────────────────────────────
restaurant_categories = ['Biryani', 'Pizza', 'Chinese', 'Fast Food',
                          'South Indian', 'Beverages', 'Desserts', 'Healthy']

# Sample customers (with replacement so some customers have multiple orders)
order_customer_ids = np.random.choice(customer_ids, size=N_ORDERS, replace=True)

# Base order value by category (INR)
category_base_values = {
    'Biryani'      : (280, 60),
    'Pizza'        : (350, 80),
    'Chinese'      : (240, 55),
    'Fast Food'    : (160, 40),
    'South Indian' : (200, 45),
    'Beverages'    : (120, 30),
    'Desserts'     : (150, 35),
    'Healthy'      : (320, 70)
}

order_categories = np.random.choice(
    restaurant_categories, size=N_ORDERS,
    p=[0.22, 0.18, 0.14, 0.16, 0.12, 0.08, 0.05, 0.05]
)

# Generate order values based on category
order_values = np.array([
    max(80, np.random.normal(
        category_base_values[cat][0],
        category_base_values[cat][1]
    ))
    for cat in order_categories
]).round(2)

# Discount percentage — discount-acquired customers get higher discounts
acq_map = customers.set_index('customer_id')['acquisition_type'].to_dict()

discount_pct = np.array([
    np.random.choice([0, 10, 20, 30, 40, 50, 60],
        p=[0.05, 0.10, 0.15, 0.20, 0.25, 0.15, 0.10])
    if acq_map.get(cid) == 'discount_acquired'
    else
    np.random.choice([0, 10, 20, 30, 40],
        p=[0.50, 0.25, 0.15, 0.07, 0.03])
    for cid in order_customer_ids
])

# Final paid amount after discount
paid_amount = (order_values * (1 - discount_pct / 100)).round(2)

# Restaurant margin (varies by category)
category_margin = {
    'Biryani': 0.32, 'Pizza': 0.38, 'Chinese': 0.28,
    'Fast Food': 0.25, 'South Indian': 0.30, 'Beverages': 0.55,
    'Desserts': 0.45, 'Healthy': 0.40
}
gross_margin = np.array([
    paid_amount[i] * category_margin[order_categories[i]]
    for i in range(N_ORDERS)
]).round(2)

# Order dates — spread across 2023, weighted after acquisition date
order_dates = pd.to_datetime('2023-01-01') + pd.to_timedelta(
    np.random.randint(0, 365, N_ORDERS), unit='D'
)

# Platform fee (Zomato charges restaurant ~20-25%)
platform_fee = (order_values * np.random.uniform(0.18, 0.25, N_ORDERS)).round(2)

orders = pd.DataFrame({
    'order_id'            : [f'ORD_{str(i).zfill(6)}' for i in range(1, N_ORDERS+1)],
    'customer_id'         : order_customer_ids,
    'order_date'          : order_dates,
    'restaurant_category' : order_categories,
    'gross_order_value'   : order_values,
    'discount_pct'        : discount_pct,
    'paid_amount'         : paid_amount,
    'platform_fee'        : platform_fee,
    'gross_margin'        : gross_margin,
    'delivery_rating'     : np.random.choice([1,2,3,4,5], N_ORDERS, p=[0.05,0.08,0.15,0.42,0.30])
})

# Merge acquisition type into orders
orders = orders.merge(customers[['customer_id','acquisition_type','acquisition_date','city']],
                      on='customer_id', how='left')

print(f'Orders created : {len(orders)}')
print(orders.dtypes)

Orders created : 10000
order_id                          str
customer_id                       str
order_date             datetime64[us]
restaurant_category               str
gross_order_value             float64
discount_pct                    int64
paid_amount                   float64
platform_fee                  float64
gross_margin                  float64
delivery_rating                 int64
acquisition_type                  str
acquisition_date       datetime64[us]
city                              str
dtype: object


## Step 3 — Data Cleaning

Even synthetic data needs cleaning practice. We deliberately inject issues to fix.

In [19]:
# ── Inject realistic dirty data ──────────────────────────────────────────────
dirty_orders = orders.copy()

# 1. Inject ~2% null values in key columns
null_idx = np.random.choice(dirty_orders.index, size=200, replace=False)
dirty_orders.loc[null_idx[:100], 'delivery_rating'] = np.nan
dirty_orders.loc[null_idx[100:], 'paid_amount']     = np.nan

# 2. Inject duplicate orders (copy 50 rows)
dup_rows = dirty_orders.sample(50, random_state=42)
dirty_orders = pd.concat([dirty_orders, dup_rows], ignore_index=True)

# 3. Inject negative order values (data entry errors)
neg_idx = np.random.choice(dirty_orders.index, size=15, replace=False)
dirty_orders.loc[neg_idx, 'gross_order_value'] = -dirty_orders.loc[neg_idx, 'gross_order_value']

# 4. Inconsistent category casing
dirty_orders.loc[np.random.choice(dirty_orders.index, 100), 'restaurant_category'] = \
    dirty_orders.loc[np.random.choice(dirty_orders.index, 100), 'restaurant_category'].str.upper()

print('=== DIRTY DATA AUDIT ===')
print(f'Total rows (with duplicates) : {len(dirty_orders)}')
print(f'\nNull counts:')
print(dirty_orders.isnull().sum()[dirty_orders.isnull().sum() > 0])
print(f'\nDuplicate rows              : {dirty_orders.duplicated().sum()}')
print(f'Negative order values       : {(dirty_orders.gross_order_value < 0).sum()}')

=== DIRTY DATA AUDIT ===
Total rows (with duplicates) : 10050

Null counts:
restaurant_category    100
paid_amount            100
delivery_rating        100
dtype: int64

Duplicate rows              : 50
Negative order values       : 15


In [20]:
# ── CLEANING PIPELINE ────────────────────────────────────────────────────────
clean = dirty_orders.copy()

# Step 3a — Remove duplicates
before = len(clean)
clean  = clean.drop_duplicates(subset=['order_id'])
print(f'Duplicates removed       : {before - len(clean)}')

# Step 3b — Fix negative order values (absolute value — data entry error)
clean['gross_order_value'] = clean['gross_order_value'].abs()
print(f'Negative values fixed    : {(orders.gross_order_value < 0).sum()}')

# Step 3c — Handle nulls
# delivery_rating: fill with median per category (more accurate than global median)
clean['delivery_rating'] = clean.groupby('restaurant_category')['delivery_rating'] \
                                .transform(lambda x: x.fillna(x.median()))

# paid_amount: recalculate from source columns if null
mask = clean['paid_amount'].isnull()
clean.loc[mask, 'paid_amount'] = (
    clean.loc[mask, 'gross_order_value'] * (1 - clean.loc[mask, 'discount_pct'] / 100)
).round(2)
print(f'Nulls remaining          : {clean.isnull().sum().sum()}')

# Step 3d — Standardise category casing
clean['restaurant_category'] = clean['restaurant_category'].str.title().str.strip()

# Step 3e — Enforce correct data types
clean['order_date']        = pd.to_datetime(clean['order_date'])
clean['acquisition_date']  = pd.to_datetime(clean['acquisition_date'])
clean['discount_pct']      = clean['discount_pct'].astype(int)
clean['delivery_rating'] = clean['delivery_rating'].fillna(3).astype(int)

# Step 3f — Derived columns useful for analysis
clean['order_month']      = clean['order_date'].dt.to_period('M')
clean['acq_month']        = clean['acquisition_date'].dt.to_period('M')
clean['days_since_acq']   = (clean['order_date'] - clean['acquisition_date']).dt.days

# Discount tier
clean['discount_tier'] = pd.cut(
    clean['discount_pct'],
    bins=[-1, 0, 20, 39, 100],
    labels=['No Discount', 'Low (1–20%)', 'Mid (21–39%)', 'High (40%+)']
)

# Net revenue per order (platform perspective)
clean['net_revenue'] = (clean['platform_fee'] - 
                        clean['gross_order_value'] * clean['discount_pct'] / 100 * 0.5).round(2)

print(f'\n=== CLEAN DATA SUMMARY ===')
print(f'Total rows    : {len(clean)}')
print(f'Date range    : {clean.order_date.min().date()} to {clean.order_date.max().date()}')
print(f'Unique customers : {clean.customer_id.nunique()}')
print(f'\nDiscount tier distribution:')
print(clean['discount_tier'].value_counts())

Duplicates removed       : 50
Negative values fixed    : 0
Nulls remaining          : 200

=== CLEAN DATA SUMMARY ===
Total rows    : 10000
Date range    : 2023-01-01 to 2023-12-31
Unique customers : 2918

Discount tier distribution:
discount_tier
Low (1–20%)     3119
High (40%+)     3101
No Discount     2284
Mid (21–39%)    1496
Name: count, dtype: int64


In [21]:
# Save cleaned data for next notebooks
os.makedirs('../data', exist_ok=True)
clean.to_csv('../data/02_cleaned_orders.csv', index=False)
customers.to_csv('../data/01_customers.csv', index=False)
print('Files saved to ../data/')

Files saved to ../data/
